# Свечи Мосбиржи: акции и фьючерсы

Переиспользуемая версия ноутбука Тимура `moexalgo_test3.ipynb` (T34). Все функции — в `scripts/moex_data.py`: один файл, нужны только pandas и интернет, библиотека moexalgo не нужна. Файл можно положить рядом с ноутбуком и работать вне репозитория.

* CSV в том же формате, что наша история `data/raw/*_1m_15-26.csv`: `open, close, high, low, value, volume, begin, end`, время московское.
* Повторный запуск продолжает с последней свечи файла: можно прервать и запустить снова.
* Фьючерсы — все контракты серии, включая истёкшие, и склейка в непрерывный ряд; позиции физлиц и юрлиц (FUTOI).
* Минутки за 10 лет по одной бумаге — порядка получаса (ISS отдаёт по 500 свечей за запрос).

То же из командной строки: `python scripts/moex_data.py shares SBER --from 2015-01-01`, `python scripts/moex_data.py futures SILV --from 2020-01-01 --interval 60 --continuous`. Для работы стратегии в реальном времени — `scripts/fetch_moex_iss.py` и Т-Инвестиции; этот ноутбук — для истории.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

HERE = Path.cwd()
# в репозитории — scripts/moex_data.py; вне его — положите moex_data.py рядом с ноутбуком
ROOT = HERE.parent if (HERE.parent / "scripts" / "moex_data.py").exists() else HERE
sys.path.insert(0, str(ROOT / "scripts"))
import moex_data as md

OUT = ROOT / "data" / "raw" / "moex"   # куда класть файлы
OUT

## 1. Что торгуется сейчас

In [ ]:
shares = md.tickers("shares")          # акции основного режима TQBR: код, название, лот, шаг цены
print(f"акций: {len(shares)}")
shares.head(10)

In [ ]:
fut = md.tickers("futures")            # фьючерсы: assetcode — код серии для md.futures_series
fut[fut["assetcode"].isin(["SILV", "GOLD", "Si", "BR"])].sort_values(["assetcode", "lasttradedate"])

## 2. Акции: свечи с докачкой

Интервалы: 1, 10, 60 минут; 24 — день. Для полной истории — `START = "2015-01-01"`.

In [ ]:
TICKERS = ["SBER", "GAZP", "LKOH"]
START = "2026-10-01"
INTERVAL = 1

for t in TICKERS:
    f = md.download(t, START, interval=INTERVAL, out=OUT / f"{t}_{md.NAMES[INTERVAL]}.csv", quiet=True)
    print(md.check(f))

## 3. Фьючерсы: все контракты серии, включая истёкшие

Коды серий: `SILV` — серебро, `GOLD` — золото, `Si` — доллар/рубль, `BR` — Brent. Каждый контракт качается только пока он ближний: от экспирации предыдущего (с запасом 15 дней на переход) до своей. У старых контрактов код с годом (`SVZ5_2015`) — его берёт `futures_series`.

In [ ]:
ASSET = "SILV"
series = md.futures_series(ASSET)
print(f"{ASSET}: контрактов {len(series)}, первый истёк {series['expiration_date'].min():%Y-%m-%d}")
series.tail(8)

In [ ]:
windows = md.download_futures(ASSET, "2025-01-01", interval=60, folder=OUT / ASSET)
windows[["secid", "since", "till"]]

## 4. Непрерывный ряд

Переход на следующий контракт — за `roll_days` дней до экспирации ближнего. Прошлое поправлено пропорционально на разрыв цены при переходе: доходности верные, уровни цен в прошлом — нет (`adjust=False` — без поправки). Колонка `contract` — какой контракт стоит в ряду.

In [ ]:
cont = md.continuous(OUT / ASSET, ASSET, interval=60, roll_days=5, series=windows)
cont.to_csv(OUT / f"{ASSET}_1h_cont.csv", index=False)
print(md.check(cont))
cont.groupby("contract", sort=False)["begin"].agg(["min", "max", "count"])

In [ ]:
cont.set_index("begin")["close"].plot(figsize=(10, 3), title=f"{ASSET}: склеенный ряд, часы");

## 5. Дописать свежие свечи к готовой истории

Перекачивать историю не нужно: `download` дописывает в конец файла только свечи позже последней. Файл должен быть ровно с колонками `open … end` — у `data/raw/GAZP_1m_15-26_upd.csv` лишняя колонка-индекс, в него дописывать нельзя (функция откажет). Работайте с копией.

In [ ]:
# import shutil
# shutil.copy(ROOT / "data" / "raw" / "SBER_1m_15-26.csv", OUT / "SBER_1m_full.csv")
# md.download("SBER", "2015-01-01", out=OUT / "SBER_1m_full.csv")
# md.check(OUT / "SBER_1m_full.csv")

## 6. Позиции физлиц и юрлиц по фьючерсам (FUTOI)

Каждые 5 минут, с 2020 года: чистая позиция `pos`, длинные и короткие, число лиц — отдельно для физлиц (`FIZ`) и юрлиц (`YUR`). Тикер — префикс серии: `sv` — серебро, `gd` — золото, `br`, `si`. Бесплатно — с задержкой 14 дней. Российский аналог отчёта CFTC о позициях (S040 в банке источников), только чаще. Первый запуск за 6 лет — около 2 тысяч запросов (по дню), порядка 15 минут.

In [ ]:
f = md.futoi("sv", "2026-09-01", out=OUT / "futoi_sv.csv")
oi = pd.read_csv(f)
oi["time"] = pd.to_datetime(oi["tradedate"] + " " + oi["tradetime"])
oi.pivot_table(index="time", columns="clgroup", values="pos").plot(figsize=(10, 3), title="SV: чистая позиция физлиц и юрлиц, контрактов");